In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [19]:
load_dotenv()

# LLM Configuration - DeepSeek R1 Distill Qwen 7B
LLAMA_STUDIO_API_BASE = os.getenv("LLAMA_STUDIO_API_BASE", "http://localhost:1234/v1")
LLAMA_STUDIO_API_KEY = os.getenv("LLAMA_STUDIO_API_KEY", "lm-studio")
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "deepseek-r1-distill-qwen-7b")
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.7"))
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "2048"))
LLM_TIMEOUT = int(os.getenv("LLM_TIMEOUT", "60"))  # Timeout in seconds
LLM_MAX_RETRIES = int(os.getenv("LLM_MAX_RETRIES", "2"))  # Number of retries



llm = ChatOpenAI(
    base_url=LLAMA_STUDIO_API_BASE,
    api_key=LLAMA_STUDIO_API_KEY,
    model=LLM_MODEL_NAME,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    timeout=LLM_TIMEOUT,
    max_retries=LLM_MAX_RETRIES,
)


In [20]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [21]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [22]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [23]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [24]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor?\n\nBecause it had too many cheese *headaches*! 😂🍕',
 'explanation': 'The joke plays on a common phrase, "having a headache," and twists it by replacing "headache" with "cheese headaches." Here\'s a breakdown:\n\n1. **Literal Interpretation**: Normally, when someone says they have a "headache," they mean pain in their head. However, the humor comes from substituting this meaning with something more specific to pizza.\n\n2. **Contextual Twist**: The joke sets up an expectation that the pizza is experiencing some kind of health issue (like going to a doctor), but instead of having a physical problem, it\'s dealing with too much cheese. Cheese can indeed be overwhelming in large quantities on a pizza!\n\n3. **Wordplay**: The humor comes from the clever wordplay where "cheese headaches" sounds like a real condition but is actually just an exaggeration about the amount of cheese.\n\nSo, the punchline combines a familiar phrase 

In [25]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it had too many cheese *headaches*! 😂🍕', 'explanation': 'The joke plays on a common phrase, "having a headache," and twists it by replacing "headache" with "cheese headaches." Here\'s a breakdown:\n\n1. **Literal Interpretation**: Normally, when someone says they have a "headache," they mean pain in their head. However, the humor comes from substituting this meaning with something more specific to pizza.\n\n2. **Contextual Twist**: The joke sets up an expectation that the pizza is experiencing some kind of health issue (like going to a doctor), but instead of having a physical problem, it\'s dealing with too much cheese. Cheese can indeed be overwhelming in large quantities on a pizza!\n\n3. **Wordplay**: The humor comes from the clever wordplay where "cheese headaches" sounds like a real condition but is actually just an exaggeration about the amount of cheese.\n\nSo, the punchline combines

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it had too many cheese *headaches*! 😂🍕', 'explanation': 'The joke plays on a common phrase, "having a headache," and twists it by replacing "headache" with "cheese headaches." Here\'s a breakdown:\n\n1. **Literal Interpretation**: Normally, when someone says they have a "headache," they mean pain in their head. However, the humor comes from substituting this meaning with something more specific to pizza.\n\n2. **Contextual Twist**: The joke sets up an expectation that the pizza is experiencing some kind of health issue (like going to a doctor), but instead of having a physical problem, it\'s dealing with too much cheese. Cheese can indeed be overwhelming in large quantities on a pizza!\n\n3. **Wordplay**: The humor comes from the clever wordplay where "cheese headaches" sounds like a real condition but is actually just an exaggeration about the amount of cheese.\n\nSo, the punchline combine

In [27]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling *noodle*!',
 'explanation': 'The humor in this joke comes from a wordplay on the word "noodle," which can refer to both the curly, thin strands of pasta and the part of the body that is responsible for thinking. The setup of the joke leads you to expect a physical reason why the pasta would go to a doctor, but instead, it plays on the double meaning of "noodle." \n\nSo, when you hear "Because it was feeling *noodle*!" the humor arises from the unexpected and clever twist that connects the physical state (feeling noodley) with the body part (the brain or head), creating a pun.'}

In [28]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling *noodle*!', 'explanation': 'The humor in this joke comes from a wordplay on the word "noodle," which can refer to both the curly, thin strands of pasta and the part of the body that is responsible for thinking. The setup of the joke leads you to expect a physical reason why the pasta would go to a doctor, but instead, it plays on the double meaning of "noodle." \n\nSo, when you hear "Because it was feeling *noodle*!" the humor arises from the unexpected and clever twist that connects the physical state (feeling noodley) with the body part (the brain or head), creating a pun.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe17-25fc-6d57-8002-fab6a10ec426'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-10-23T07:25:26.986867+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '',

In [29]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling *noodle*!', 'explanation': 'The humor in this joke comes from a wordplay on the word "noodle," which can refer to both the curly, thin strands of pasta and the part of the body that is responsible for thinking. The setup of the joke leads you to expect a physical reason why the pasta would go to a doctor, but instead, it plays on the double meaning of "noodle." \n\nSo, when you hear "Because it was feeling *noodle*!" the humor arises from the unexpected and clever twist that connects the physical state (feeling noodley) with the body part (the brain or head), creating a pun.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe17-25fc-6d57-8002-fab6a10ec426'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-10-23T07:25:26.986867+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': ''

### Time Travel

In [30]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f0afe16-a73b-61fd-8000-1ae3f6d730db"}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f0afe16-a73b-61fd-8000-1ae3f6d730db'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-10-23T07:25:13.695483+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe16-a73a-632f-bfff-ec5128929a54'}}, tasks=(PregelTask(id='a84a9ef8-deab-a628-36f6-6679815f691e', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling *noodle*!'}),), interrupts=())

In [31]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f0afe16-a73b-61fd-8000-1ae3f6d730db"}})

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy?\n\nBecause it had too many noodles to deal with! 😊🍝',
 'explanation': 'The joke plays on a play of words, specifically the double meaning of "noodles." In this case, "noodles" refers to both the strands of pasta and the word that means "ideas or thoughts," which are often discussed in therapy. The humor comes from the pun: the pasta is seeking therapy not because it\'s feeling emotional distress, but because it has so many strands (or "noodles") that it can\'t handle them all!'}

In [32]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy?\n\nBecause it had too many noodles to deal with! 😊🍝', 'explanation': 'The joke plays on a play of words, specifically the double meaning of "noodles." In this case, "noodles" refers to both the strands of pasta and the word that means "ideas or thoughts," which are often discussed in therapy. The humor comes from the pun: the pasta is seeking therapy not because it\'s feeling emotional distress, but because it has so many strands (or "noodles") that it can\'t handle them all!'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe22-6cd8-6006-8002-dc849f570659'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-10-23T07:30:29.695674+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe22-1d22-6bdc-8001-72faaab1062a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'j

#### Updating State

In [ ]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

In [ ]:
list(workflow.get_state_history(config1))

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

In [ ]:
list(workflow.get_state_history(config1))

### Fault Tolerance

In [33]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [34]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [35]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [36]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [37]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [38]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


KeyboardInterrupt: 

In [39]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe2e-494a-6086-8001-04ff45b18be2'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-10-23T07:35:48.090062+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe2e-4948-6c51-8000-3fb5f272a722'}}, tasks=(PregelTask(id='7046db4e-c002-b145-1322-87fe97e0d734', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0afe2e-4948-6c51-8000-3fb5f272a722'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-10-23T07:35:48.089550+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch